## Setup

In [1]:
import pandas as pd
import numpy as np
import os
from langchain_community.document_loaders import PyPDFLoader, UnstructuredPDFLoader, PyPDFium2Loader
from langchain.document_loaders import PyPDFDirectoryLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from pathlib import Path
import random
import ollama
import networkx as nx
import hypernetx as hnx 


## Input data directory
data_dir = "PFAS_test"
inputdirectory = Path(f"./data_input/{data_dir}")
## This is where the output csv files will be written
out_dir = data_dir
outputdirectory = Path(f"./data_output/{out_dir}")

## Load Documents

In [2]:
## Dir PDF Loader
#loader = PyPDFDirectoryLoader(inputdirectory)
## File Loader
#loader = PyPDFLoader("./data/MedicalDocuments/orf-path_health-n1.pdf")
loader = DirectoryLoader(inputdirectory, show_progress=True)
documents = loader.load()

100%|██████████| 1/1 [00:01<00:00,  1.13s/it]


## Extract Concepts

In [3]:
## This function uses the helpers/prompt function to extract concepts from text
from helpers.df_helpers import docs2Graph
from helpers.df_helpers import docs2Hypergraph

If regenerate is set to True then the dataframes are regenerated and Both the dataframes are written in the csv format so we dont have to calculate them again. 

        dfne = dataframe of edges

        df = dataframe of chunks


Else the dataframes are read from the output directory

In [ ]:
# To regenerate the graph with LLM, set this to True
regenerate = False

if regenerate:
    # Generate concepts list from full text documents (no chunking)
   # G_graph = docs2Graph(documents, model='codellama:13b-instruct')
    G_graph = docs2Graph(documents, model='mistral:instruct')
    # Convert to DataFrame from the graph
    dfg1 = nx.to_pandas_edgelist(G_graph)
    #save the graph
    nx.write_graphml(G_graph, "output_graph.graphml")

else: 
    G_graph = nx.read_graphml("output_graph.graphml")
    dfg1 = nx.to_pandas_edgelist(G_graph)

Generating triples...
 ```
{
  "nodes": [
    {
      "id": "PDMS",
      "name": "PDMS"
    },
    {
      "id": "carbon_black",
      "name": "carbon black"
    },
    {
      "id": "iron(II)_sulfate",
      "name": "iron(II) sulfate"
    },
    {
      "id": "hydrogen_peroxide",
      "name": "hydrogen peroxide"
    },
    {
      "id": "pH_buffering_agents",
      "name": "pH buffering agents"
    },
    {
      "id": "Zinc_oxide",
      "name": "Zinc oxide"
    },
    {
      "id": "lactic_acid",
      "name": "lactic acid"
    },
    {
      "id": "Polyethylene_oxide",
      "name": "Polyethylene oxide"
    },
    {
      "id": "lithium_salt",
      "name": "lithium salt"
    },
    {
      "id": "ceramic_nanoparticles",
      "name": "ceramic nanoparticles"
    },
    {
      "id": "plasticizers",
      "name": "plasticizers"
    },
    {
      "id": "chitosan",
      "name": "chitosan"
    },
    {
      "id": "genipin",
      "name": "genipin"
    }
  ],
  "edges": [
    {
   

In [ ]:
import pickle
import pandas as pd

# To regenerate the hypergraph with LLM, set this to True
regenerate = False

if regenerate:
    # Generate hypergraph from full text documents
    H_hypergraph = docs2Hypergraph(documents, model='mistral:instruct')
    
    # Save as pickle (recommended format for HyperNetX)
    with open("output_hypergraph.pkl", "wb") as f:
        pickle.dump(H_hypergraph, f)

    # Optional: convert hyperedges to DataFrame
    dfg1 = pd.DataFrame([
        {"event": hedge, "entity": node}
        for hedge, nodes in H_hypergraph.incidence_dict.items()
        for node in nodes
    ])

    # Save DataFrame for inspection
    dfg1.to_csv("output_hypergraph_edges.csv", index=False)

else:
    # Load from pickle
    with open("output_hypergraph.pkl", "rb") as f:
        H_hypergraph = pickle.load(f)

    # Reconstruct DataFrame
    dfg1 = pd.DataFrame([
        {"event": hedge, "entity": node}
        for hedge, nodes in H_hypergraph.incidence_dict.items()
        for node in nodes
    ])

Generating hypergraph events...
 ```
{
  "events": [
    {
      "id": "PDMS and carbon black combine to form a flexible, piezoresistive matrix",
      "entities": ["PDMS", "carbon black"]
    },
    {
      "id": "Interplay between iron(II) sulfate, hydrogen peroxide, and pH buffering agents during oxidative crosslinking of matrix proteins in engineered bone grafts",
      "entities": ["iron(II) sulfate", "hydrogen peroxide", "pH buffering agents", "matrix proteins"]
    },
    {
      "id": "Zinc oxide and lactic acid promote antibacterial activity and osteoinductivity in bioresorbable scaffold coatings",
      "entities": ["zinc oxide", "lactic acid", "antibacterial activity", "osteoinductivity"]
    },
    {
      "id": "Polyethylene oxide, lithium salt, ceramic nanoparticles, plasticizers produce a solid polymer electrolyte for biosensing applications in smart bone implants",
      "entities": ["polyethylene oxide", "lithium salt", "ceramic nanoparticles", "plasticizers"]
    },
 

In [5]:
print(type(G_graph))
print(f"Generated graph with {G_graph.number_of_nodes()} nodes and {G_graph.number_of_edges()} edges.")

NameError: name 'G_graph' is not defined

### Calculate communities for coloring the nodes

In [10]:
communities_generator = nx.community.girvan_newman(G_graph)
top_level_communities = next(communities_generator)
next_level_communities = next(communities_generator)
communities = sorted(map(sorted, next_level_communities))
print("Number of Communities = ", len(communities))
print(communities)

Number of Communities =  7
[['PDMS', 'carbon_black'], ['Polyethylene_oxide', 'ceramic_nanoparticles', 'plasticizers'], ['Zinc_oxide', 'lactic_acid'], ['chitosan', 'genipin'], ['hydrogen_peroxide'], ['iron(II)_sulfate', 'pH_buffering_agents'], ['lithium_salt']]


### Create a dataframe for community colors

In [11]:
import seaborn as sns
palette = "hls"

## Now add these colors to communities and make another dataframe
def colors2Community(communities) -> pd.DataFrame:
    ## Define a color palette
    p = sns.color_palette(palette, len(communities)).as_hex()
    random.shuffle(p)
    rows = []
    group = 0
    for community in communities:
        color = p.pop()
        group += 1
        for node in community:
            rows += [{"node": node, "color": color, "group": group}]
    df_colors = pd.DataFrame(rows)
    return df_colors


colors = colors2Community(communities)
colors

,node,color,group
0,PDMS,#db57c0,1
1,carbon_black,#db57c0,1
2,Polyethylene_oxide,#75db57,2
3,ceramic_nanoparticles,#75db57,2
4,plasticizers,#75db57,2
5,Zinc_oxide,#57dbaa,3
6,lactic_acid,#57dbaa,3
7,chitosan,#579bdb,4
8,genipin,#579bdb,4
9,hydrogen_peroxide,#dbd057,5


### Add colors to the graph

In [12]:
for index, row in colors.iterrows():
    G_graph.nodes[row['node']]['group'] = row['group']
    G_graph.nodes[row['node']]['color'] = row['color']
    G_graph.nodes[row['node']]['size'] = G_graph.degree[row['node']]

In [1]:
from pyvis.network import Network

graph_output_directory = "./docs/index.html"

net = Network(
    notebook=False,
    # bgcolor="#1a1a1a",
    cdn_resources="remote",
    height="900px",
    width="100%",
    select_menu=True,
    # font_color="#cccccc",
    filter_menu=False,
)

net.from_nx(G_graph)
# net.repulsion(node_distance=150, spring_length=400)
net.force_atlas_2based(central_gravity=0.015, gravity=-31)
# net.barnes_hut(gravity=-18100, central_gravity=5.05, spring_length=380)
net.show_buttons(filter_=["physics"])

net.show(graph_output_directory, notebook=False)

NameError: name 'G_graph' is not defined